# 05_connectivity_results_explainer.ipynb

## Ecological Connectivity Results Interpretation Report

This notebook translates model outputs into scientifically interpretable results for conservation planning and reporting.

---

## Audience
Scientific director / conservation decision-makers

## Goal
- Interpret connectivity outputs
- Compare species patterns
- Identify ecological corridors and bottlenecks
- Produce publication-ready visual summaries


## Imports

In [1]:
from pathlib import Path
import rasterio
import numpy as np
import matplotlib.pyplot as plt

import geopandas as gpd


## Project Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

CONNECTIVITY_DIR = PROJECT_ROOT / 'data' / 'processed' / 'connectivity'
SPECIES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'species'

print(CONNECTIVITY_DIR)


/home/linda/Documents/myData/data-management/data/processed/connectivity


## Define Species List (Study Set)

In [3]:
species_list = [
    'Emys_orbicularis',
    'Canis_lupus',
    'Lynx_pardinus',
    'Vulpes_vulpes'
]


## Load Connectivity Layers

In [4]:
flow_maps = {}

for sp in species_list:
    path = CONNECTIVITY_DIR / sp / 'current_flow.tif'
    
    if path.exists():
        with rasterio.open(path) as src:
            flow_maps[sp] = src.read(1)
    else:
        print(f'Missing: {sp}')


Missing: Emys_orbicularis
Missing: Canis_lupus
Missing: Lynx_pardinus
Missing: Vulpes_vulpes


## Standardized Visualization Function

In [5]:
def plot_flow(raster, title, cmap='magma'):
    plt.figure(figsize=(8, 6))
    plt.imshow(raster, cmap=cmap)
    plt.colorbar(label='Connectivity intensity')
    plt.title(title)
    plt.axis('off')
    plt.show()


## Species Connectivity Overview

In [6]:
for sp, raster in flow_maps.items():
    plot_flow(raster, f'{sp} - Connectivity Flow')


## Normalized Comparison Across Species

In [7]:
normalized = {}

for sp, raster in flow_maps.items():
    arr = raster.astype(float)
    arr = (arr - np.min(arr)) / (np.max(arr) - np.min(arr) + 1e-9)
    normalized[sp] = arr


## Multi-Species Mean Connectivity Surface

In [8]:
stack = np.stack(list(normalized.values()), axis=0)
mean_connectivity = np.mean(stack, axis=0)

plt.figure(figsize=(10, 8))
plt.imshow(mean_connectivity, cmap='viridis')
plt.colorbar(label='Mean Connectivity')
plt.title('Multi-Species Connectivity Hotspots')
plt.axis('off')
plt.show()


ValueError: need at least one array to stack

## Ecological Interpretation

### Key Interpretation Principles

- High values = frequent movement pathways or low resistance corridors
- Low values = ecological barriers or unsuitable matrix
- Overlap across species = multi-species conservation priority zones

### What this means ecologically

- Dark/high-flow corridors represent landscape permeability
- These are candidate ecological networks
- These zones should be prioritized for:
  - habitat restoration
  - protection zoning
  - connectivity conservation policy


## Corridor Consensus (Hotspot Stability)

In [ ]:
# Identify persistent corridors across species
threshold = 0.6

consensus = (mean_connectivity > threshold).astype(int)

plt.figure(figsize=(10, 8))
plt.imshow(consensus, cmap='Greens')
plt.title('Multi-Species Corridor Consensus')
plt.axis('off')
plt.show()


## Scientific Summary for Reporting

### Key Findings (automated interpretation layer)

1. Connectivity is spatially heterogeneous across the study region.
2. Certain corridors are consistently important across multiple species.
3. High-consensus zones represent robust ecological linkages.
4. These areas are priority candidates for conservation interventions.

---

### Management Implications

- Protect high-consensus corridors from fragmentation
- Restore degraded connectivity gaps
- Prioritize multi-species overlap zones for Natura 2000 reinforcement


## Export Summary Layer

In [ ]:
import geopandas as gpd
from shapely.geometry import box

# Convert raster grid to simple bounding box footprint (for QGIS reference)
gdf = gpd.GeoDataFrame(
    {'value': [1]},
    geometry=[box(0, 0, mean_connectivity.shape[1], mean_connectivity.shape[0])],
    crs='EPSG:3035'
)

summary_path = PROJECT_ROOT / 'outputs' / 'connectivity_summary.gpkg'
gdf.to_file(summary_path, driver='GPKG')

print('Saved:', summary_path)


## Final Statement

This workflow translates resistance-based ecological modeling into spatially explicit conservation intelligence.

It supports:
- landscape planning
- Natura 2000 optimization
- multi-species corridor conservation
- scenario-based restoration design
